# Inngang analyse

Innledende deskriptiv analyse, kode fra Eirik.


In [45]:
from snowflake.snowpark.functions import col, sum as sum_, max as max_, datediff, round as round_, year, month, when, lit, lag, avg, count, stddev, to_date, dayofweek, weekofyear, dayofmonth, quarter, last_day
from snowflake.snowpark import Window
import re
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# from dwh_tools.get_or_create_session import get_or_create_session
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session
import os
import requests
import gzip
import json
# from dwh_tools.get_or_create_session import get_or_create_session
pd.set_option('display.max_columns', None)
import time
import pywin



In [46]:
def get_or_create_session(schema: str = None) -> Session:
    """
    Returns the active Snowpark session if running inside Snowflake,
    otherwise creates one locally using Azure OAuth (interactive login if needed).
 
    Parameters
    ----------
    config_path : str
        Path to a JSON config file containing Snowflake connection parameters
        (account, warehouse, role, database, schema).
 
    Returns
    -------
    Session
        A Snowpark Session object.
    """
    # If running on snowflake
    if 'POSIT_PRODUCT' in os.environ:
        session = Session.builder.getOrCreate()
        session.sql("USE DATABASE PROD_FOR_SKADE_PRODUKT_ADHOC").collect()
        if schema:
            session.sql("USE SCHEMA " + schema).collect()
        else:
            session.sql("USE SCHEMA PRODUKT_WRITE_DEV").collect()
        session.sql("USE WAREHOUSE SKADE_VWH").collect()
 
        return session
 
    try:
        session = get_active_session()
        return session
    except Exception:
        import win32api
        if schema is None:
            schema = 'PRODUKT_WRITE_DEV'
 
        connection_parameters = {
            "server": "km28161.west-europe.azure.snowflakecomputing.com",
            "warehouse": "SKADE_VWH",
            "account": "VK82539-KLP",
            "database": "PROD_FOR_SKADE_PRODUKT_ADHOC",
            "schema" : schema,
            "user": win32api.GetUserNameEx(win32api.NameUserPrincipal),  
            "authenticator": "externalbrowser"
        }
       
        # Create the session
        session = Session.builder.configs(connection_parameters).create()
        return session

In [47]:
session = get_or_create_session()

In [48]:
def plot_mean_and_count(
    df: pd.DataFrame,
    x_col: str,
    mean_col: str,
    max_unique: int = 30,
    title: str | None = None,
) -> go.Figure:
    """Dual-axis Plotly chart: bar = row count, line = mean of mean_col.

    Parameters
    ----------
    df        : source DataFrame
    x_col     : column to group/bin along x-axis
    mean_col  : column whose mean is plotted on the right y-axis
    max_unique: if the number of unique values in x_col exceeds this threshold
                numeric values are rounded to a nice step size so the axis stays
                numeric (e.g. ages rounded to nearest 5 or 10)
    title     : optional chart title (defaults to "<mean_col> by <x_col>")
    """
    BAR_COLOR = "#7EB8D4"   # muted steel blue
    LINE_COLOR = "#E07B54"  # warm terracotta

    series = df[x_col].dropna()
    n_unique = series.nunique()

    if pd.api.types.is_numeric_dtype(series) and n_unique > max_unique:
        raw_step = (series.max() - series.min()) / max_unique
        # Round step up to nearest "nice" magnitude (1, 2, 5, 10, 20, 50, ...)
        magnitude = 10 ** np.floor(np.log10(raw_step))
        nice_step = magnitude * np.ceil(raw_step / magnitude)
        x_rounded = (df[x_col] / nice_step).round() * nice_step
        grouped = df.assign(**{x_col: x_rounded}).groupby(x_col, observed=True).agg(
            count=(x_col, "size"),
            mean_val=(mean_col, "mean"),
        )
    else:
        grouped = df.groupby(x_col, observed=True).agg(
            count=(x_col, "size"),
            mean_val=(mean_col, "mean"),
        )

    grouped = grouped.sort_index()
    x_labels = grouped.index.tolist()

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=x_labels,
        y=grouped["count"],
        name="Count",
        marker_color=BAR_COLOR,
        opacity=0.75,
        yaxis="y1",
    ))

    fig.add_trace(go.Scatter(
        x=x_labels,
        y=grouped["mean_val"],
        name=f"Mean {mean_col}",
        mode="lines+markers",
        line=dict(color=LINE_COLOR, width=2.5),
        marker=dict(size=6),
        yaxis="y2",
    ))

    fig.update_layout(
        title=title or f"{mean_col} by {x_col}",
        xaxis=dict(title=x_col, tickangle=-35),
        yaxis=dict(title="Count", showgrid=True, gridcolor="#EBEBEB"),
        yaxis2=dict(
            title=f"Mean {mean_col}",
            overlaying="y",
            side="right",
            showgrid=False,
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        plot_bgcolor="white",
        paper_bgcolor="white",
        bargap=0.15,
    )

    return fig




In [49]:
df = session.table('elh_write.inngangsdata').to_pandas()
df.columns = df.columns.str.lower()

df_info = session.table('inngangsdata_info').to_pandas()
df_info.columns = df_info.columns.str.lower()


In [50]:
df['ankomst_dato'] = pd.to_datetime(df['ankomst_dato'], format="%d.%m.%Y %H:%M:%S").dt.date
df['behandlingstid'] = (
    df['behandlingstid']
        .str.split(':')
            .apply(lambda x: int(x[0]) * 60 + int(x[1]))
            )

df_info['hf_dato'] = pd.to_datetime(df_info['hf_dato'], format="%Y-%m-%d").dt.date

In [58]:
df.head()

,unique_id,forste_kogruppe,forste_ko,kunde,ankomst_dato,tid_i_ko,behandlet,behandlingstid,etterbehandlingstid,behandlingstid_etterbehandlingstid,overfort_eksternt,arbeidsplass,agent_id,god_reserve
0,3051408,SK_Salg_Ekst_PM,SK_Salg_Ekst_PM_Tlf,99525091,2026-06-02,3,Behandlet,92,01:29,03:01,None,wp3409,lni,God agent
1,3051415,SK_Salg_Ny_PM,SK_Salg_Ny_PM_Tlf,0046707954838,2026-06-02,16,Behandlet,456,00:47,08:23,None,wp3404,edm,God agent
2,3051412,SK_Salg_Ekst_PM,SK_Salg_Ekst_PM_Tlf,97322088,2026-06-02,3,Behandlet,598,00:00,09:58,None,wp3403,ba9,God agent
3,3051410,SK_Salg_Ekst_PM,SK_Salg_Ekst_PM_Tlf,92433365,2026-06-02,86,Behandlet,382,03:00,09:22,None,wp3342,njn,God agent
4,3051403,SK_Salg_Ekst_PM,SK_Salg_Ekst_PM_Tlf,91521443,2026-06-02,34,Behandlet,200,01:30,04:50,None,wp3409,lni,God agent


In [51]:
df_ = pd.merge(df, df_info, left_on='ankomst_dato', right_on='hf_dato', how='left')

In [52]:
df_.columns

Index(['unique_id', 'forste_kogruppe', 'forste_ko', 'kunde', 'ankomst_dato',
       'tid_i_ko', 'behandlet', 'behandlingstid', 'etterbehandlingstid',
       'behandlingstid_etterbehandlingstid', 'overfort_eksternt',
       'arbeidsplass', 'agent_id', 'god_reserve', 'hf_dato',
       'antall_hf_b30_upr01_ny', 'antall_hf_b7_upr01_ny',
       'antall_hf_f30_upr01_ny', 'antall_hf_f7_upr01_ny',
       'antall_hf_b30_upr01_for', 'stddev_premieendring_b30_upr01_for',
       'snitt_premieendring_b30_upr01_for', 'antall_hf_b7_upr01_for',
       'stddev_premieendring_b7_upr01_for', 'snitt_premieendring_b7_upr01_for',
       'antall_hf_f30_upr01_for', 'stddev_premieendring_f30_upr01_for',
       'snitt_premieendring_f30_upr01_for', 'antall_hf_f7_upr01_for',
       'stddev_premieendring_f7_upr01_for', 'snitt_premieendring_f7_upr01_for',
       'antall_hf_b30_eph01_ny', 'antall_hf_b7_eph01_ny',
       'antall_hf_f30_eph01_ny', 'antall_hf_f7_eph01_ny',
       'antall_hf_b30_eph01_for', 'stddev_premi

In [53]:
plot_mean_and_count(df_,
                    x_col='maaned',
                    mean_col='behandlingstid',
                    max_unique=60)